## 1. Persiapan Environment
Mengimpor *library* yang diwajibkan: Polars untuk *lazy evaluation* dan *pipeline* transformasi, serta DuckDB untuk *query* analitik.

In [5]:
import polars as pl
import duckdb
import time

# Mendefinisikan Lokasi file input (CSV berita) dan output (Parquet)
CSV_PATH = "../data/raw/data.csv"
PARQUET_PATH = "../data/raw/indonesian_news.parquet"

## 2. Transformasi ke Parquet dengan Polars (Lazy Evaluation)
Menggunakan `scan_csv` untuk membaca dataset raksasa tanpa membebani RAM (Lazy Evaluation), lalu mengonversinya ke format Parquet agar proses *profiling* dan analisis selanjutnya berjalan jauh lebih efisien.

In [6]:
print("Memulai proses Lazy Evaluation dengan Polars...")
start_time = time.time()

# 1. Lazy load: Tidak memuat seluruh data ke RAM, hanya membuat pipeline rencana eksekusi
lazy_df = pl.scan_csv(CSV_PATH, ignore_errors=True)

# 2. Sink: Mengeksekusi pipeline dan langsung menuliskannya ke format Parquet
print(f"Mengonversi file CSV ke Parquet... (Mohon tunggu)")
lazy_df.sink_parquet(PARQUET_PATH)

print(f"Konversi selesai dalam {time.time() - start_time:.2f} detik!")
print(f"File hasil optimasi tersimpan di: {PARQUET_PATH}")

Memulai proses Lazy Evaluation dengan Polars...
Mengonversi file CSV ke Parquet... (Mohon tunggu)
Konversi selesai dalam 9.99 detik!
File hasil optimasi tersimpan di: ../data/raw/indonesian_news.parquet


## 3. Data Profiling menggunakan DuckDB
Mengeksekusi perintah `SUMMARIZE` dari DuckDB secara langsung ke file Parquet untuk mendapatkan ringkasan statistik deskriptif (tipe data, nilai min/max, jumlah *null*, dan kardinalitas unik).

In [7]:
print("Mengeksekusi DuckDB SUMMARIZE...")
start_time = time.time()

# Menjalankan query SUMMARIZE langsung ke file Parquet (sangat cepat dan hemat RAM)
query = f"SUMMARIZE SELECT * FROM read_parquet('{PARQUET_PATH}')"

# Mengambil hasilnya sebagai DataFrame agar tampilannya rapi di Jupyter
df_summary = duckdb.query(query).df()

print(f"Waktu eksekusi Profiling DuckDB: {time.time() - start_time:.2f} detik")

# Menampilkan tabel hasil profiling
display(df_summary)

Mengeksekusi DuckDB SUMMARIZE...
Waktu eksekusi Profiling DuckDB: 3.19 detik


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,id,BIGINT,33,63567,32684,34081.74877042921,18044.068638414126,17745,35146,49818,32735,0.00
1,source,VARCHAR,cnbcindonesia,tempo,6,NaN,NaN,NaN,NaN,NaN,32735,0.00
2,title,VARCHAR,Daftar Panglima Kostrad Dari Masa ke Masa,​​Tips Atasi Masalah Jerawat dengan YOUr Acne ...,30501,NaN,NaN,NaN,NaN,NaN,32735,0.00
3,image,VARCHAR,,https://statik.tempo.co/data/2023/04/12/id_119...,35667,NaN,NaN,NaN,NaN,NaN,32735,0.00
4,url,VARCHAR,https://bisnis.tempo.co/read/1702047/syarat-da...,https://www.suara.com/news/2023/04/12/060000/b...,35724,NaN,NaN,NaN,NaN,NaN,32735,0.00
5,content,VARCHAR,,zakat terdapat di dalam rukun Islam. Hal ini m...,33646,NaN,NaN,NaN,NaN,NaN,32735,0.00
6,date,VARCHAR,2018-08-10 05:11:58+00,2023-04-11 20:13:25+00,33885,NaN,NaN,NaN,NaN,NaN,32735,0.00
7,embedding,VARCHAR,"[-0.000100255646,-0.00392203,0.009619573,-0.01...","[9.975993e-05,-0.027434079,0.010739251,-0.0164...",33604,NaN,NaN,NaN,NaN,NaN,32735,0.00
8,created_at,VARCHAR,2023-03-03 16:08:03.144472,2023-04-11 20:30:13.315332,3026,NaN,NaN,NaN,NaN,NaN,32735,0.00
9,updated_at,VARCHAR,2023-03-03 16:08:03.144472,2023-04-11 20:30:13.315332,3026,NaN,NaN,NaN,NaN,NaN,32735,0.00


## 4. Pengecekan Missing Values (Polars Expressions)
Memanfaatkan Polars *lazy evaluation* dan *expressions* untuk memindai seluruh kolom dan menghitung anomali atau data kosong (*null*).

In [9]:
# Mendapatkan daftar nama kolom
# Membaca file parquet secara lazy menggunakan Polars
lazy_pq = pl.scan_parquet(PARQUET_PATH)
schema = lazy_pq.collect_schema()

# Menyusun ekspresi pengecekan null/kosong secara aman berdasarkan tipe data
exprs = []
for col_name, dtype in schema.items():
    # Jika kolom bertipe string/utf8, cek null atau string kosong
    if dtype == pl.Utf8 or dtype == pl.String:
        expr = (pl.col(col_name).is_null() | (pl.col(col_name) == "") | (pl.col(col_name).str.strip_chars() == "")).sum().alias(col_name)
    else:
        # Untuk kolom non-string (angka/dll), cukup cek nilai null saja
        expr = pl.col(col_name).is_null().sum().alias(col_name)
    exprs.append(expr)

# Eksekusi perhitungan
null_or_empty_counts = lazy_pq.select(exprs).collect()

print("Jumlah Missing Values atau Data Kosong:")
display(null_or_empty_counts)

Jumlah Missing Values atau Data Kosong:


id,source,title,image,url,content,date,embedding,created_at,updated_at,summary
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,160,0,441,0,0,0,0,10
